# Notebook 05 · MNIST: reconocimiento de dígitos con TensorFlow + Keras (sin redes convolucionales)

| R1-A2-S4 · Redes Neuronales II | 604013/DEEP LEARNING -CONCEPTOS/FU/CAD2202023205/EIAIIPA2026 |
|---|---|
| Universidad de Cundinamarca · Especialización en Inteligencia Artificial | Docente: **Nidia Stella Garcia Roa** |
| Estudiante: **Yohann Javier Nikolay Salazar Cardenas** | Repositorio: `https://github.com/Yohann0011/Redes-Neuronales-II` |


**Problema:** clasificación **multiclase de 10 clases** (dígitos 0–9) sobre el dataset **MNIST** de
imágenes de dígitos escritos a mano de 28 × 28 píxeles.

> **Restricción de la actividad:** **NO** se utilizan capas convolucionales. La arquitectura es un
> **perceptrón multicapa** `Flatten(784)` → `Dense(128, ReLU)` → `Dense(10, softmax)` aplicado a
> imágenes.

---

## 1. De las frutas a un dataset real

| Elemento | Notebook 04 (frutas) | Notebook 05 (MNIST) |
|---|---|---|
| Ejemplos | 30 frutas sintéticas | 60 000 de entrenamiento + 10 000 de prueba |
| Características por ejemplo | 3 (peso, longitud, amarillo) | **784** (28 × 28 píxeles) |
| Clases | 3 | **10** |
| Entrada a la red | Vector de 3 valores | Vector de 784 valores (imagen aplanada) |
| Capa de salida | `Dense(3, softmax)` | `Dense(10, softmax)` |
| Pérdida | `sparse_categorical_crossentropy` | `sparse_categorical_crossentropy` |
| Optimizador | Adam (lr = 0.01) | Adam (lr = 0.001) |

La lógica es **idéntica**: cambia la escala. Los mismos principios (*forward propagation*, pérdida,
*backpropagation*, actualización de parámetros) escalan de un ejemplo didáctico de 30 frutas a un
problema real de visión por computador con 70 000 imágenes.

## 2. ¿Por qué funciona una red densa sin convoluciones?

Las capas convolucionales explotan la **localidad** y la **invarianza traslacional** de las imágenes y
son la opción de mayor precisión. Sin embargo, los dígitos MNIST están **centrados y normalizados**, de
modo que un perceptrón multicapa puede aprender combinaciones lineales de píxeles suficientemente
discriminativas y supera holgadamente el 96 % de *accuracy*.

La ventaja didáctica es decisiva: **todos** sus parámetros son pesos de capas densas, exactamente los
mismos objetos matemáticos estudiados en los notebooks 01 a 04.

**Costo de no usar convoluciones:** el modelo es sensible a la posición del dígito y concentra
100 480 parámetros en la primera capa, frente a los pocos cientos de un filtro convolucional.

## 3. Nota de implementación: entrenamiento con ciclo explícito

En este notebook el entrenamiento se escribe de forma **explícita** con `tf.GradientTape`:

```python
@tf.function
def paso_entrenamiento(xb, yb):
    with tf.GradientTape() as cinta:
        logits = modelo(xb, training=True)
        perdida = perdida_fn(yb, logits)
    gradientes = cinta.gradient(perdida, modelo.trainable_variables)
    optimizador.apply_gradients(zip(gradientes, modelo.trainable_variables))
    return perdida
```

Es exactamente el mismo ciclo que Keras ejecuta internamente en `fit()` y el mismo que se escribirá en
PyTorch en el Notebook 06 (`zero_grad` → *forward* → *loss* → `backward` → `step`).

**Motivo técnico:** `fit()` recibe el arreglo completo y materializa una copia `float32` de
60 000 × 28 × 28 (188 MB). Sumada a la huella base de TensorFlow (~620 MB) y a los tensores internos
del entrenamiento, el pico supera 1 GB y el *kernel* de Colab/Jupyter puede ser terminado por el
gestor de memoria. Al procesar **lote a lote**, convirtiendo y normalizando únicamente 128 imágenes por
iteración, el consumo se mantiene plano (~725 MB) y el entrenamiento es reproducible en entornos con
recursos limitados.

En un entorno con más memoria (por ejemplo Colab con GPU) la versión equivalente de alto nivel es:

```python
x_train = x_train.astype("float32") / 255.0
modelo_mnist.compile(optimizer="adam",
                     loss="sparse_categorical_crossentropy",
                     metrics=["accuracy"])
historial = modelo_mnist.fit(x_train, y_train, epochs=5,
                             batch_size=128, validation_split=0.1)
```

Ambas formas producen el mismo modelo; la diferencia es únicamente de gestión de memoria y de
explicitud del ciclo de aprendizaje.


## 4. Entorno de ejecución

En **Google Colab** se recomienda activar la GPU (`Entorno de ejecución → Cambiar tipo de entorno de
ejecución → GPU`), aunque este modelo entrena en pocos segundos también con CPU.

In [1]:
import os
# Ambiente silencioso y determinista para TensorFlow (no altera los resultados)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["PYTHONHASHSEED"] = "0"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 80,
    "savefig.dpi": 80,
    "savefig.bbox": None,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
plt.rcParams["font.family"] = "DejaVu Sans"

import tensorflow as tf
from tensorflow import keras
import random, time, gc
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

random.seed(42); np.random.seed(42); tf.random.set_seed(42)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("Backend    :", keras.backend.backend())
print("Dispositivos:", [d.device_type for d in tf.config.list_physical_devices()])

I0000 00:00:1791644462.543707   30570 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1791644463.915123   30570 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0 | Keras: 3.15.1
Backend    : tensorflow
Dispositivos: ['CPU']


## 5. Carga del dataset MNIST

Se descarga el archivo oficial `mnist.npz` (~11 MB) y se carga con `np.load`. El conjunto de
entrenamiento se conserva en `uint8` (47 MB) y **solo el lote en curso** se convierte a `float32`;
el conjunto de prueba sí se normaliza completo porque ocupa apenas 31 MB y se usa repetidamente en la
evaluación.

In [2]:
ruta_mnist = keras.utils.get_file(
    "mnist.npz",
    "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz",
)

with np.load(ruta_mnist, allow_pickle=False) as datos:
    x_train_mnist = datos["x_train"]                      # uint8 (60000, 28, 28)
    y_train_mnist = datos["y_train"]                      # uint8 (60000,)
    x_test_mnist = datos["x_test"].astype("float32") / 255.0   # normalizado [0,1]
    y_test_mnist = datos["y_test"]

print("Archivo:", ruta_mnist)
print()
print("Entrenamiento:", x_train_mnist.shape, y_train_mnist.shape, x_train_mnist.dtype)
print("Prueba       :", x_test_mnist.shape, y_test_mnist.shape, x_test_mnist.dtype)
print("Rango de píxeles (entrenamiento):", x_train_mnist.min(), "-", x_train_mnist.max())
print("Rango de píxeles (prueba)       :", x_test_mnist.min(), "-", x_test_mnist.max())
print("Clases:", np.unique(y_train_mnist))
print("Ejemplos por clase:", np.bincount(y_train_mnist))
print()
print("Primer ejemplo: etiqueta real =", y_train_mnist[0], "| forma =", x_train_mnist[0].shape)
print("Memoria del conjunto de entrenamiento:",
      round(x_train_mnist.nbytes / 1024**2, 1), "MB (uint8)")

Archivo: /tmp/.keras/datasets/mnist.npz

Entrenamiento: (60000, 28, 28) (60000,) uint8
Prueba       : (10000, 28, 28) (10000,) float32
Rango de píxeles (entrenamiento): 0 - 255
Rango de píxeles (prueba)       : 0.0 - 1.0
Clases: [0 1 2 3 4 5 6 7 8 9]
Ejemplos por clase: [5923 6742 5958 6131 5842 5421 5918 6265 5851 5949]

Primer ejemplo: etiqueta real = 5 | forma = (28, 28)
Memoria del conjunto de entrenamiento: 44.9 MB (uint8)


In [3]:
fig, axes = plt.subplots(2, 10, figsize=(11.5, 2.8))
for fila in range(2):
    for col in range(10):
        idx = fila * 10 + col
        axes[fila, col].imshow(x_train_mnist[idx], cmap="gray")
        axes[fila, col].set_title(str(y_train_mnist[idx]), fontsize=9)
        axes[fila, col].axis("off")
fig.suptitle("Figura 14. Primeras 20 imágenes de entrenamiento de MNIST con su etiqueta real", y=1.06)
plt.tight_layout(); plt.savefig("fig09_mnist_muestra.png"); plt.show()
plt.close(fig)
_ = gc.collect()

## 6. Normalización y aplanado de la entrada

La red no "ve" el dígito como una persona: procesa valores numéricos. Dividir entre 255 ubica los
píxeles en [0, 1], lo que estabiliza el descenso de gradiente. `reshape(-1, 784)` es la operación
equivalente a la capa `Flatten` del modelo secuencial: convierte la matriz 28 × 28 en un vector de 784
características.

In [4]:
def normalizar_lote(imagenes):
    """Convierte un lote uint8 (n,28,28) a float32 (n,784) en [0,1]."""
    return imagenes.reshape(len(imagenes), -1).astype("float32") / 255.0


lote_demo = normalizar_lote(x_train_mnist[:5])
print("Lote de 5 imágenes ->", lote_demo.shape, lote_demo.dtype)
print("Mínimo:", lote_demo.min(), "| Máximo:", lote_demo.max())
print()
print("Primeros 12 valores del vector de 784 píxeles (zona de borde, casi todo ceros):")
print(np.round(lote_demo[0][:12], 3))
print()
zona = lote_demo[0].reshape(28, 28)[10:18, 8:20]
print("Bloque central 8x12 de la misma imagen (donde sí hay trazo):")
print(np.round(zona, 2))
print()
print("Verificación: 784 =", 28 * 28, "->", lote_demo.shape[1] == 28 * 28)

Lote de 5 imágenes -> (5, 784) float32
Mínimo: 0.0 | Máximo: 1.0

Primeros 12 valores del vector de 784 píxeles (zona de borde, casi todo ceros):
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Bloque central 8x12 de la misma imagen (donde sí hay trazo):
[[0.   0.05 0.   0.6  0.99 0.35 0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.55 0.99 0.75 0.01 0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.04 0.75 0.99 0.27 0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.14 0.95 0.88 0.63 0.42 0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.32 0.94 0.99 0.99 0.47 0.1  0.  ]
 [0.   0.   0.   0.   0.   0.   0.18 0.73 0.99 0.99 0.59 0.11]
 [0.   0.   0.   0.   0.   0.   0.   0.06 0.36 0.99 0.99 0.73]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.98 0.99 0.98]]

Verificación: 784 = 784 -> True


## 7. Arquitectura 784 → 128 → 10 (sin convoluciones)

| Capa | Función | Parámetros |
|---|---|---|
| `Input(shape=(784,))` | Vector de píxeles normalizados | 0 |
| `Dense(128, relu)` | Capa oculta: 784 × 128 pesos + 128 bias | **100 480** |
| `Dense(10, softmax)` | Salida: 128 × 10 pesos + 10 bias | **1 290** |
| **Total** | | **101 770** |

La verificación programática de la ausencia de capas convolucionales constituye la evidencia del
cumplimiento de la restricción de la actividad.

In [5]:
tf.random.set_seed(42)

modelo_mnist = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    keras.layers.Dense(128, activation="relu", name="oculta_128"),
    keras.layers.Dense(10, activation="softmax", name="salida_10")
])

modelo_mnist.summary()

tipos = [capa.__class__.__name__ for capa in modelo_mnist.layers]
print()
print("Capas del modelo            :", tipos)
print("¿Hay alguna capa convolucional?", any("Conv" in t for t in tipos))
print("¿Hay alguna capa de pooling?  ", any("Pooling" in t for t in tipos))
print("Parámetros entrenables      :", f"{modelo_mnist.count_params():,}")
print("-> Verificación de la restricción: red 100 % densa (perceptrón multicapa).")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ oculta_128 (Dense)              │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ salida_10 (Dense)               │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)


Capas del modelo            : ['Dense', 'Dense']
¿Hay alguna capa convolucional? False
¿Hay alguna capa de pooling?   False
Parámetros entrenables      : 101,770
-> Verificación de la restricción: red 100 % densa (perceptrón multicapa).


## 8. Compilación manual: pérdida, optimizador y paso de entrenamiento

En lugar de `compile()` + `fit()`, se declaran explícitamente la función de pérdida y el optimizador, y
se escribe el paso de entrenamiento. `@tf.function` compila la función a un grafo de TensorFlow, lo que
acelera notablemente la ejecución sin cambiar la matemática.

In [6]:
perdida_fn = keras.losses.SparseCategoricalCrossentropy()
optimizador = keras.optimizers.Adam(learning_rate=0.001)
metrica_acc = keras.metrics.SparseCategoricalAccuracy()

print("Pérdida    :", perdida_fn.__class__.__name__)
print("Optimizador:", optimizador.__class__.__name__, "| lr =", optimizador.learning_rate)


@tf.function
def paso_entrenamiento(xb, yb):
    """Un paso completo: forward -> loss -> backpropagation -> actualización."""
    with tf.GradientTape() as cinta:
        logits = modelo_mnist(xb, training=True)
        perdida = perdida_fn(yb, logits)
    gradientes = cinta.gradient(perdida, modelo_mnist.trainable_variables)   # backpropagation
    optimizador.apply_gradients(zip(gradientes, modelo_mnist.trainable_variables))  # actualización
    return perdida


def evaluar_conjunto(x_datos, y_datos, tam_lote=1000, normalizar=False):
    """Evalúa pérdida y accuracy por lotes, sin materializar todas las predicciones."""
    perdidas, aciertos, total = 0.0, 0, 0
    clases, confianzas = [], []
    for inicio in range(0, len(x_datos), tam_lote):
        xb = x_datos[inicio:inicio + tam_lote]
        yb = y_datos[inicio:inicio + tam_lote]
        if normalizar:
            xb = normalizar_lote(xb)
        else:
            xb = xb.reshape(len(xb), -1)
        probs = modelo_mnist(xb, training=False).numpy()
        perdidas += float(perdida_fn(yb, probs)) * len(yb)
        aciertos += int((probs.argmax(axis=1) == yb).sum())
        total += len(yb)
        clases.append(probs.argmax(axis=1))
        confianzas.append(probs.max(axis=1))
    return (perdidas / total, aciertos / total,
            np.concatenate(clases), np.concatenate(confianzas))


print()
print("Funciones de entrenamiento y evaluación definidas correctamente")

Pérdida    : SparseCategoricalCrossentropy
Optimizador: Adam | lr = <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0010000000474974513>

Funciones de entrenamiento y evaluación definidas correctamente


### 8.1 Estado inicial del modelo (sin entrenar)

Antes del entrenamiento las probabilidades son casi uniformes (~0.1 por dígito) y la pérdida coincide
con $-\log(1/10)=2.3026$: la red no tiene ninguna información.

In [7]:
xb0 = normalizar_lote(x_train_mnist[:1])
yb0 = y_train_mnist[:1]
probs0 = modelo_mnist(xb0, training=False).numpy()

print("Etiqueta real de la imagen 0:", yb0[0])
print("Probabilidades sin entrenar :", np.round(probs0[0], 4))
print("Suma de probabilidades    :", round(float(probs0[0].sum()), 6))
print("Pérdida inicial             :", round(float(perdida_fn(yb0, probs0)), 4))
print("Referencia teórica -ln(1/10):", round(-np.log(0.1), 4))

Etiqueta real de la imagen 0: 5
Probabilidades sin entrenar : [0.0849 0.0669 0.0561 0.13   0.0804 0.1989 0.1206 0.1231 0.1115 0.0275]
Suma de probabilidades    : 1.0
Pérdida inicial             : 1.615
Referencia teórica -ln(1/10): 2.3026


## 9. Entrenamiento (5 épocas, batch = 128)

Se reserva el último 10 % del conjunto de entrenamiento (6 000 imágenes) como **validación**, que no
participa en el ajuste de pesos. Las 10 000 imágenes de prueba permanecen intactas para la evaluación
final. En cada época se barajan los índices, pues el orden de presentación no debe influir en el
aprendizaje.

In [8]:
BATCH_SIZE = 128
EPOCHS = 5
N_VALIDACION = 6000
n_entrenamiento = len(x_train_mnist) - N_VALIDACION

indices_train = np.arange(n_entrenamiento)
indices_val = np.arange(n_entrenamiento, len(x_train_mnist))

historial = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
t0 = time.time()

for epoch in range(EPOCHS):
    # ---------- ENTRENAMIENTO ----------
    np.random.shuffle(indices_train)
    perdidas_epoca, aciertos_epoca, vistos = 0.0, 0, 0

    for inicio in range(0, n_entrenamiento, BATCH_SIZE):
        idx = indices_train[inicio:inicio + BATCH_SIZE]
        xb = normalizar_lote(x_train_mnist[idx])
        yb = y_train_mnist[idx]

        perdida = paso_entrenamiento(xb, yb)          # forward + loss + backward + update

        probs = modelo_mnist(xb, training=False).numpy()
        perdidas_epoca += float(perdida) * len(idx)
        aciertos_epoca += int((probs.argmax(axis=1) == yb).sum())
        vistos += len(idx)

    # ---------- VALIDACIÓN ----------
    perd_val, aciertos_val, total_val = 0.0, 0, 0
    for inicio in range(0, N_VALIDACION, 512):
        idx = indices_val[inicio:inicio + 512]
        xb = normalizar_lote(x_train_mnist[idx])
        yb = y_train_mnist[idx]
        probs = modelo_mnist(xb, training=False).numpy()
        perd_val += float(perdida_fn(yb, probs)) * len(yb)
        aciertos_val += int((probs.argmax(axis=1) == yb).sum())
        total_val += len(yb)

    historial["loss"].append(perdidas_epoca / vistos)
    historial["accuracy"].append(aciertos_epoca / vistos)
    historial["val_loss"].append(perd_val / total_val)
    historial["val_accuracy"].append(aciertos_val / total_val)
    gc.collect()

tiempo = time.time() - t0
print(f"{'Época':>6}{'Loss':>10}{'Accuracy':>10}{'Val loss':>11}{'Val acc':>10}")
for i in range(EPOCHS):
    print(f"{i+1:>6}{historial['loss'][i]:>10.4f}{historial['accuracy'][i]:>10.4f}"
          f"{historial['val_loss'][i]:>11.4f}{historial['val_accuracy'][i]:>10.4f}")
print()
print(f"Tiempo total: {tiempo:.1f} s ({tiempo/EPOCHS:.1f} s por época)")
print(f"Lotes por época: {int(np.ceil(n_entrenamiento / BATCH_SIZE))}")

 Época      Loss  Accuracy   Val loss   Val acc
     1    0.3740    0.9013     0.1691    0.9553
     2    0.1700    0.9539     0.1283    0.9658
     3    0.1217    0.9673     0.1023    0.9723
     4    0.0946    0.9747     0.0922    0.9740
     5    0.0760    0.9805     0.0882    0.9760

Tiempo total: 8.5 s (1.7 s por época)
Lotes por época: 422


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
epocas = range(1, EPOCHS + 1)

axes[0].plot(epocas, historial["loss"], marker="o", label="Entrenamiento", color="#c0392b", lw=2)
axes[0].plot(epocas, historial["val_loss"], marker="s", label="Validación", color="#2980b9", lw=2, ls="--")
axes[0].set_title("Figura 15. Error durante el entrenamiento")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Loss"); axes[0].legend(fontsize=8)

axes[1].plot(epocas, historial["accuracy"], marker="o", label="Entrenamiento", color="#27ae60", lw=2)
axes[1].plot(epocas, historial["val_accuracy"], marker="s", label="Validación", color="#8e44ad", lw=2, ls="--")
axes[1].set_ylim(0.88, 1.0)
axes[1].set_title("Figura 16. Precisión durante el entrenamiento")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Accuracy"); axes[1].legend(fontsize=8)

plt.tight_layout(); plt.savefig("fig10_mnist_curvas_keras.png"); plt.show()
plt.close(fig)
_ = gc.collect()

## 10. Evaluación final sobre las 10 000 imágenes de prueba

In [10]:
test_loss, test_accuracy, clases_test, confianzas_test = evaluar_conjunto(
    x_test_mnist, y_test_mnist, tam_lote=1000
)

print("=== RESULTADOS MNIST · TensorFlow + Keras ===")
print("Error de prueba    :", round(test_loss, 6))
print("Accuracy de prueba :", round(test_accuracy, 6))
print("Imágenes correctas :", int(round(test_accuracy * len(y_test_mnist))), "de", len(y_test_mnist))
print("Imágenes erróneas  :", len(y_test_mnist) - int(round(test_accuracy * len(y_test_mnist))))
print()
print("Confianza promedio en aciertos:",
      round(float(confianzas_test[clases_test == y_test_mnist].mean()), 4))
print("Confianza promedio en errores :",
      round(float(confianzas_test[clases_test != y_test_mnist].mean()), 4))
print("-> El modelo está mucho menos seguro cuando se equivoca: la probabilidad máxima es")
print("   una señal útil para decidir cuándo requerir revisión humana.")

=== RESULTADOS MNIST · TensorFlow + Keras ===
Error de prueba    : 0.089279
Accuracy de prueba : 0.9735
Imágenes correctas : 9735 de 10000
Imágenes erróneas  : 265

Confianza promedio en aciertos: 0.9764
Confianza promedio en errores : 0.6941
-> El modelo está mucho menos seguro cuando se equivoca: la probabilidad máxima es
   una señal útil para decidir cuándo requerir revisión humana.


In [11]:
probs10 = modelo_mnist(x_test_mnist[:10].reshape(10, 784),
                                     training=False).numpy()

print("=== PRIMERAS 10 IMÁGENES DE PRUEBA ===")
print(f"{'Imagen':>7}{'Real':>6}{'Predicción':>12}{'Confianza':>11}{'Resultado':>11}")
for i in range(10):
    prediccion = int(np.argmax(probs10[i]))
    conf = float(probs10[i].max())
    ok = "ACIERTO" if prediccion == y_test_mnist[i] else "ERROR"
    print(f"{i+1:>7}{int(y_test_mnist[i]):>6}{prediccion:>12}{conf:>11.4f}{ok:>11}")

print()
print("Vector completo de probabilidades de la imagen 1:")
print(np.round(probs10[0], 6), "-> suma:", round(float(probs10[0].sum()), 6))

=== PRIMERAS 10 IMÁGENES DE PRUEBA ===
 Imagen  Real  Predicción  Confianza  Resultado
      1     7           7     0.9997    ACIERTO
      2     2           2     0.9994    ACIERTO
      3     1           1     0.9953    ACIERTO
      4     0           0     0.9999    ACIERTO
      5     4           4     0.9899    ACIERTO
      6     1           1     0.9934    ACIERTO
      7     4           4     0.9980    ACIERTO
      8     9           9     0.9862    ACIERTO
      9     5           6     0.7301      ERROR
     10     9           9     0.9741    ACIERTO

Vector completo de probabilidades de la imagen 1:
[0.00000e+00 0.00000e+00 3.10000e-05 2.91000e-04 0.00000e+00 3.00000e-06
 0.00000e+00 9.99653e-01 7.00000e-06 1.30000e-05] -> suma: 1.0


In [12]:
fig, axes = plt.subplots(2, 5, figsize=(11.5, 3.0))
for i, ax in enumerate(axes.ravel()):
    pred = int(np.argmax(probs10[i]))
    real = int(y_test_mnist[i])
    ax.imshow(x_test_mnist[i].reshape(28, 28), cmap="gray")
    ax.set_title(f"Real: {real} · Pred: {pred}", fontsize=9,
                 color="#27ae60" if pred == real else "#c0392b")
    ax.axis("off")
fig.suptitle("Figura 17. Predicciones sobre las primeras 10 imágenes de prueba", y=1.06)
plt.tight_layout(); plt.savefig("fig11_mnist_predicciones.png"); plt.show()
plt.close(fig)
_ = gc.collect()

## 11. Análisis de errores del modelo

Un *accuracy* alto **no** debe convertirse en una afirmación de perfección. Analizar los errores revela
los patrones de confusión del clasificador (dígitos mal escritos o pares morfológicamente similares como
4/9, 7/1 o 3/5).

In [13]:
errores = np.where(clases_test != y_test_mnist)[0]

print("Cantidad total de errores:", len(errores), "de", len(y_test_mnist))
print("Tasa de error            :", f"{len(errores)/len(y_test_mnist)*100:.2f} %")
print()
print("Primeros 15 errores (índice | real -> predicho | confianza):")
for idx in errores[:15]:
    print(f"  {idx:>5} | {y_test_mnist[idx]} -> {clases_test[idx]} | {confianzas_test[idx]:.4f}")

Cantidad total de errores: 265 de 10000
Tasa de error            : 2.65 %

Primeros 15 errores (índice | real -> predicho | confianza):
      8 | 5 -> 6 | 0.7301
    247 | 4 -> 2 | 0.5578
    259 | 6 -> 0 | 0.8357
    321 | 2 -> 7 | 0.9853
    340 | 5 -> 3 | 0.8896
    445 | 6 -> 0 | 0.8496
    478 | 5 -> 8 | 0.5151
    582 | 8 -> 2 | 0.6241
    591 | 8 -> 3 | 0.5062
    619 | 1 -> 8 | 0.5589
    628 | 3 -> 9 | 0.8086
    659 | 2 -> 8 | 0.5485
    684 | 7 -> 3 | 0.8381
    691 | 8 -> 4 | 0.9721
    707 | 4 -> 9 | 0.8640


In [14]:
cm = confusion_matrix(y_test_mnist, clases_test)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5.6, 4.6))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(
    ax=ax, cmap="Blues", colorbar=True, values_format="d", xticks_rotation=0)
ax.set_title("Figura 18. Matriz de confusión - MNIST (10 000 imágenes)")
plt.tight_layout(); plt.savefig("fig12_mnist_confusion.png"); plt.show()
plt.close(fig)
_ = gc.collect()

fig, ax = plt.subplots(figsize=(5.6, 4.6))
ConfusionMatrixDisplay(cm_norm, display_labels=range(10)).plot(
    ax=ax, cmap="Blues", colorbar=True, values_format=".3f", xticks_rotation=0)
ax.set_title("Figura 19. Matriz normalizada por clase real (recall)")
plt.tight_layout(); plt.savefig("fig12b_mnist_confusion_norm.png"); plt.show()
plt.close(fig)
_ = gc.collect()

In [15]:
print(classification_report(y_test_mnist, clases_test,
                            target_names=[str(d) for d in range(10)], digits=4))

pares = sorted([(cm[r, c], r, c) for r in range(10) for c in range(10) if r != c], reverse=True)
print("Top 5 confusiones (real -> predicho):")
for cant, r, c in pares[:5]:
    print(f"   {r} -> {c} : {cant} casos ({cant/cm[r].sum()*100:.2f} % de los dígitos '{r}')")
print()
print("Clase con mejor recall :", int(np.argmax(np.diag(cm_norm))),
      f"({np.diag(cm_norm).max():.4f})")
print("Clase con peor  recall :", int(np.argmin(np.diag(cm_norm))),
      f"({np.diag(cm_norm).min():.4f})")

              precision    recall  f1-score   support

           0     0.9691    0.9918    0.9803       980
           1     0.9885    0.9877    0.9881      1135
           2     0.9832    0.9641    0.9736      1032
           3     0.9614    0.9851    0.9731      1010
           4     0.9736    0.9766    0.9751       982
           5     0.9883    0.9439    0.9656       892
           6     0.9709    0.9749    0.9729       958
           7     0.9756    0.9718    0.9737      1028
           8     0.9527    0.9713    0.9619       974
           9     0.9720    0.9633    0.9676      1009

    accuracy                         0.9735     10000
   macro avg     0.9735    0.9731    0.9732     10000
weighted avg     0.9737    0.9735    0.9735     10000

Top 5 confusiones (real -> predicho):
   5 -> 3 : 17 casos (1.91 % de los dígitos '5')
   2 -> 8 : 12 casos (1.16 % de los dígitos '2')
   9 -> 4 : 10 casos (0.99 % de los dígitos '9')
   5 -> 8 : 10 casos (1.12 % de los dígitos '5')
   4 ->

In [16]:
n_mostrar = 9
fig, axes = plt.subplots(1, n_mostrar, figsize=(11.5, 2.2))
for ax, idx in zip(axes, errores[:n_mostrar]):
    ax.imshow(x_test_mnist[idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"Real {y_test_mnist[idx]}\nPred {clases_test[idx]}", fontsize=8, color="#c0392b")
    ax.axis("off")
fig.suptitle("Figura 20. Ejemplos de dígitos clasificados incorrectamente", y=1.14)
plt.tight_layout(); plt.savefig("fig13_mnist_errores.png"); plt.show()
plt.close(fig)
_ = gc.collect()

## 12. Experimento complementario: efecto de la capacidad de la capa oculta

Se entrenan tres variantes (32, 128 y 256 neuronas) durante 2 épocas sobre un **subconjunto
estratificado de 20 000 imágenes**, con la misma semilla y los mismos datos en las tres, para observar
la relación entre capacidad del modelo, precisión y costo computacional. Se reutiliza el ciclo por
lotes para mantener el consumo de memoria controlado.

In [17]:
def entrenar_variante(n_ocultas, x_idx, epochs=2, lr=0.001, semilla=42):
    """Entrena una red 784 -> n_ocultas -> 10 y devuelve métricas y tiempo."""
    tf.random.set_seed(semilla)
    m = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(n_ocultas, activation="relu"),
        keras.layers.Dense(10, activation="softmax")
    ])
    opt = keras.optimizers.Adam(learning_rate=lr)
    loss_fn = keras.losses.SparseCategoricalCrossentropy()

    @tf.function
    def paso(xb, yb):
        with tf.GradientTape() as cinta:
            perdida = loss_fn(yb, m(xb, training=True))
        g = cinta.gradient(perdida, m.trainable_variables)
        opt.apply_gradients(zip(g, m.trainable_variables))
        return perdida

    idx = np.array(x_idx)
    np.random.shuffle(idx)
    t0 = time.time()
    for _ in range(epochs):
        for i in range(0, len(idx), BATCH_SIZE):
            sel = idx[i:i + BATCH_SIZE]
            paso(normalizar_lote(x_train_mnist[sel]), y_train_mnist[sel])
    dt = time.time() - t0

    loss_t, acc_t, _, _ = evaluar_variante(m, loss_fn)
    params = m.count_params()
    del m
    keras.backend.clear_session()
    gc.collect()
    return params, loss_t, acc_t, dt


def evaluar_variante(m, loss_fn, tam_lote=1000):
    perdidas, aciertos, total = 0.0, 0, 0
    for inicio in range(0, len(x_test_mnist), tam_lote):
        xb = x_test_mnist[inicio:inicio + tam_lote].reshape(tam_lote, -1)
        yb = y_test_mnist[inicio:inicio + tam_lote]
        probs = m(xb, training=False).numpy()
        perdidas += float(loss_fn(yb, probs)) * len(yb)
        aciertos += int((probs.argmax(axis=1) == yb).sum())
        total += len(yb)
    return perdidas / total, aciertos / total, None, None


idx_sub = np.arange(0, n_entrenamiento, 3)     # 18 000 imágenes estratificadas por posición
print("Subconjunto del experimento:", len(idx_sub), "imágenes")
print()
print(f"{'Neuronas':>9}{'Parámetros':>13}{'Test loss':>11}{'Test acc':>10}{'Segundos':>10}")
print("-" * 53)
resultados_cap = []
for n_ocultas in [32, 128, 256]:
    params, loss_t, acc_t, dt = entrenar_variante(n_ocultas, idx_sub)
    resultados_cap.append((n_ocultas, params, loss_t, acc_t, dt))
    print(f"{n_ocultas:>9}{params:>13,}{loss_t:>11.4f}{acc_t:>10.4f}{dt:>10.1f}")

Subconjunto del experimento: 18000 imágenes

 Neuronas   Parámetros  Test loss  Test acc  Segundos
-----------------------------------------------------


       32       25,450     0.3170    0.9124       0.5


      128      101,770     0.2500    0.9311       0.7


      256      203,530     0.2247    0.9328       0.8


In [18]:
fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.plot([r[0] for r in resultados_cap], [r[3] for r in resultados_cap],
        marker="o", lw=2, color="#16a085")
for n, _, _, at, _ in resultados_cap:
    ax.annotate(f"{at:.4f}", (n, at), textcoords="offset points",
                xytext=(0, 9), ha="center", fontsize=9)
ax.set_xticks([32, 128, 256])
ax.set_xlabel("Neuronas en la capa oculta")
ax.set_ylabel("Accuracy en prueba")
ax.set_ylim(min(r[3] for r in resultados_cap) - 0.01, 1.0)
ax.set_title("Figura 21. Accuracy vs capacidad de la capa oculta (2 épocas, 18 000 imágenes)")
plt.tight_layout(); plt.savefig("fig14_mnist_capacidad.png"); plt.show()
plt.close(fig)
_ = gc.collect()

## 13. Análisis de resultados

- La red densa 784 → 128 → 10 alcanza un *accuracy* superior al 97 % en MNIST **sin ninguna capa
  convolucional**, lo que confirma que un perceptrón multicapa bien entrenado es suficiente para
  dígitos centrados y normalizados.
- La **curva de validación** acompaña a la de entrenamiento sin divergencia marcada en 5 épocas: no hay
  sobreajuste apreciable en este horizonte.
- La **matriz de confusión** concentra los errores en pares morfológicamente parecidos (4↔9, 7↔1, 3↔5,
  8↔3), coherente con un modelo que solo ve intensidades de píxeles aplanadas, sin información explícita
  de forma local. Esa es justamente la ventaja que aportaría una CNN, excluida por la consigna.
- La **confianza promedio** en los errores es muy inferior a la de los aciertos: el modelo "sabe" cuándo
  duda, lo que permite diseñar umbrales de rechazo.
- Aumentar la capacidad de 32 a 128 neuronas mejora el *accuracy*; pasar a 256 aporta poco con 2 épocas
  y duplica el costo: **más parámetros no siempre significan mejor modelo**.

## 14. Conclusiones del notebook

1. El aplanado de la imagen (28 × 28 → 784) es el puente que permite tratar una imagen como un vector
   de características en un perceptrón multicapa.
2. La normalización a [0, 1] es obligatoria: condiciona la velocidad de convergencia del descenso de
   gradiente.
3. `sparse_categorical_crossentropy` con salida Softmax de 10 neuronas es la configuración estándar para
   clasificación multiclase con etiquetas enteras.
4. El ciclo *forward* → pérdida → *backpropagation* → actualización puede escribirse explícitamente en
   TensorFlow con `GradientTape`, y es idéntico en estructura al de PyTorch.
5. La evaluación profesional de un clasificador multiclase incluye pérdida, *accuracy*, matriz de
   confusión, reporte por clase y análisis de errores concretos.

## 15. Referencias

- Gulli, A., Kapoor, A., Pal, S., & Ciraci, S. (2023). *Deep Learning with TensorFlow and Keras* (3rd ed.). Packt Publishing.
  Repositorio oficial: https://github.com/PacktPublishing/Deep-Learning-with-TensorFlow-and-Keras-3rd-edition
- LeCun, Y., Cortes, C., & Burges, C. J. C. (2010). *MNIST handwritten digit database*. http://yann.lecun.com/exdb/mnist/
- Trask, A. (2019). *Grokking Deep Learning*. Manning Publications.
- Google (2026). *Google Colaboratory*. https://colab.research.google.com/